# 1 load package and data

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import pandas as pd
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryo"
h5ad_file = Path(folder_path) / "nt_nmp_nc_2026_02_06.h5ad"
nt= sc.read_h5ad(h5ad_file)

# 2.Extended Data Fig. 4a

In [ ]:
# 假设您的 t-SNE 结果保存在 `adata.obsm` 中，并命名为 'X_tsne'
tsne_coords =nt.obsm['X_umap']

# 提取 `leiden` 聚类信息，假设在 `adata.obs` 中有 'leiden' 列
leiden_clusters = nt.obs['leiden_hvg_1']

import pandas as pd
# 创建 DataFrame，包含 t-SNE 坐标和 `leiden` 信息
tsne_df = pd.DataFrame(tsne_coords, columns=['tSNE_1', 'tSNE_2'])
tsne_df['leiden_hvg_1'] = leiden_clusters.values

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='tSNE_1', y='tSNE_2', hue='leiden_hvg_1', palette=custom_palette, s=5, edgecolor=None)
plt.title("t-SNE Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\umap_heatmap\umap_nt_hvg.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

# 3.Extended Data Fig. 4d,e

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.interpolate import make_interp_spline
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter1d

def _intersections_on_grid(x, y1, y2):
    """ x """
    d = y1 - y2
    s = np.sign(d)
    idx = np.where(np.diff(s) != 0)[0]    # 
    xs = []
    for i in idx:
        x0, x1 = x[i], x[i+1]
        d0, d1 = d[i], d[i+1]
        if np.isfinite(d0) and np.isfinite(d1) and (d1 != d0):
            xs.append(x0 - d0 * (x1 - x0) / (d1 - d0))
    return xs

def plot_binned_density(
    adata,
    coord_col='AP',                 #  ( 'AP', 'DV', 'LR')
    group_col='leiden_1',           #  ( 'leiden_1', 'cell_type')
    n_bins=50,
    palette=None,
    figsize=(15, 6),
    xlim=None,
    ylim=None,
    fill=True,
    fill_alpha=0.5,                 # 0.5
    linewidth=2.5,
    smooth=0,
    show_legend=True,
    vertical_lines=None,
    vline_color='blue',
    vline_style='--',
    vline_width=1.5,
    vline_alpha=0.8,
    density_method='pdf',
    pairs_to_draw=None,
    post_smooth_sigma=0,
    dashed_groups=None,             #  ['1', '3', '5']
    secondary_y_groups=None,        # Y ['10', '11']
    secondary_ylim=None,            # Y
    x_tick_interval=100,            # X100
    primary_y_groups=None,          # 🆕 Y ['0', '1', '2']None
):
    """
     (coord_col)  bins  (group_col) 
    
    :
    - fill_alpha: float,  (0-1)0.5
    - dashed_groups: list of str, 
    - secondary_y_groups: list of str, Y
    - secondary_ylim: tuple, Y (ymin, ymax)
    - x_tick_interval: int or float, X100
    - primary_y_groups: list of str, YNonesecondary
    """

    # —— 1) 
    plot_df = adata.obs.copy()
    
    # 
    if coord_col not in plot_df.columns:
        raise KeyError(f"obs  '{coord_col}'")
    if group_col not in plot_df.columns:
        raise KeyError(f"obs  '{group_col}'")
    
    # 
    plot_df[coord_col] = pd.to_numeric(plot_df[coord_col], errors='coerce')
    plot_df = plot_df.dropna(subset=[coord_col, group_col])
    
    if plot_df.empty:
        print("Warning: No valid data found.")
        return None, {}

    #  cluster  1 vs '1' 
    plot_df[group_col] = plot_df[group_col].astype(str)
    all_groups = sorted(plot_df[group_col].unique(), key=lambda x: (len(x), x))

    # 
    dashed_groups = [str(g) for g in dashed_groups] if dashed_groups else []
    secondary_y_groups = [str(g) for g in secondary_y_groups] if secondary_y_groups else []
    primary_y_groups = [str(g) for g in primary_y_groups] if primary_y_groups else None

    # 🆕 
    if primary_y_groups is None and secondary_y_groups:
        # secondarysecondary
        groups_to_plot = all_groups
    elif primary_y_groups is not None:
        # primary_y_groupsprimary + secondary
        groups_to_plot = list(set(primary_y_groups) | set(secondary_y_groups))
        # 
        groups_to_plot = [g for g in all_groups if g in groups_to_plot]
    else:
        # 
        groups_to_plot = all_groups

    # —— 2)  bin
    coord_min, coord_max = plot_df[coord_col].min(), plot_df[coord_col].max()
    bins = np.linspace(coord_min, coord_max, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_width = bins[1] - bins[0]

    # —— 3)  group //
    densities = {}
    for cl in groups_to_plot:  # 🆕 
        #  group_col  coord_col
        vals = plot_df.loc[plot_df[group_col] == cl, coord_col].to_numpy()

        if density_method == 'kde':
            try:
                kde = gaussian_kde(vals)
                density = kde(bin_centers)
            except Exception:
                # 
                hist, _ = np.histogram(vals, bins=bins)
                density = hist / (len(vals) * bin_width if len(vals) > 0 else 1)
        else:
            hist, _ = np.histogram(vals, bins=bins)
            if density_method == 'pdf':
                density = hist / (len(vals) * bin_width if len(vals) > 0 else 1)
            elif density_method == 'frequency':
                density = hist / bin_width
            elif density_method == 'relative':
                density = hist / (len(vals) if len(vals) > 0 else 1)
            elif density_method == 'count':
                density = hist.astype(float)
            elif density_method == 'normalized':
                m = hist.max() if hist.size > 0 else 0
                density = hist / (m + 1e-10)
            else:
                raise ValueError(f": {density_method}")

        densities[cl] = density

    # —— 3.5)  + 
    x_dense = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 20)
    y_dense = {}
    for cl in groups_to_plot:  # 🆕 
        y = densities[cl]
        y_interp = np.interp(x_dense, bin_centers, y)
        if post_smooth_sigma and post_smooth_sigma > 0 and density_method != 'kde':
            y_interp = gaussian_filter1d(y_interp, sigma=post_smooth_sigma)
        y_dense[cl] = y_interp

    # —— 3.6) 
    intersections = {}  # {(cl1, cl2): [x1, x2, ...]}
    for i, c1 in enumerate(groups_to_plot):
        for j in range(i+1, len(groups_to_plot)):
            c2 = groups_to_plot[j]
            xs = _intersections_on_grid(x_dense, y_dense[c1], y_dense[c2])
            intersections[(c1, c2)] = xs

    # —— 4)  - Y
    fig, ax = plt.subplots(figsize=figsize)
    
    # Y
    ax2 = None
    if secondary_y_groups:
        ax2 = ax.twinx()

    # 🆕 
    if primary_y_groups is not None:
        # primary_y_groups
        primary_groups = [g for g in groups_to_plot if g in primary_y_groups and g not in secondary_y_groups]
    else:
        # secondary
        primary_groups = [g for g in groups_to_plot if g not in secondary_y_groups]
    
    secondary_groups = [g for g in groups_to_plot if g in secondary_y_groups]

    # Y
    for cl in primary_groups:
        color = palette.get(cl, None) if palette else None
        y = densities[cl]
        
        # 
        linestyle = '--' if cl in dashed_groups else '-'

        if smooth > 0 and density_method != 'kde':
            try:
                x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 10)
                k = min(int(smooth), len(bin_centers) - 1, 5)
                spl = make_interp_spline(bin_centers, y, k=k)
                y_s = np.maximum(spl(x_smooth), 0)
                if fill:
                    ax.fill_between(x_smooth, y_s, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax.plot(x_smooth, y_s, linewidth=linewidth, color=color,
                       linestyle=linestyle,
                       label=f'{group_col} {cl}' if not fill else '')
            except Exception:
                if fill:
                    ax.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax.plot(bin_centers, y, linewidth=linewidth, color=color,
                       linestyle=linestyle,
                       label=f'{group_col} {cl}' if not fill else '')
        else:
            if fill:
                ax.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
            ax.plot(bin_centers, y, linewidth=linewidth, color=color,
                   linestyle=linestyle,
                   label=f'{group_col} {cl}' if not fill else '')

    # Y
    if ax2:
        for cl in secondary_groups:
            color = palette.get(cl, None) if palette else None
            y = densities[cl]
            
            linestyle = '--' if cl in dashed_groups else '-'

            if smooth > 0 and density_method != 'kde':
                try:
                    x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 10)
                    k = min(int(smooth), len(bin_centers) - 1, 5)
                    spl = make_interp_spline(bin_centers, y, k=k)
                    y_s = np.maximum(spl(x_smooth), 0)
                    if fill:
                        ax2.fill_between(x_smooth, y_s, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                    ax2.plot(x_smooth, y_s, linewidth=linewidth, color=color,
                            linestyle=linestyle,
                            label=f'{group_col} {cl}' if not fill else '')
                except Exception:
                    if fill:
                        ax2.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                    ax2.plot(bin_centers, y, linewidth=linewidth, color=color,
                            linestyle=linestyle,
                            label=f'{group_col} {cl}' if not fill else '')
            else:
                if fill:
                    ax2.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax2.plot(bin_centers, y, linewidth=linewidth, color=color,
                        linestyle=linestyle,
                        label=f'{group_col} {cl}' if not fill else '')

    # 
    if vertical_lines is not None:
        for xv in vertical_lines:
            ax.axvline(xv, color=vline_color, linestyle=vline_style,
                        linewidth=vline_width, alpha=vline_alpha, zorder=10)

    #  pair 
    if pairs_to_draw:
        for p in pairs_to_draw:
            c1, c2 = map(str, p)  # 
            xs = intersections.get((c1, c2), []) + intersections.get((c2, c1), [])
            for xv in xs:
                ax.axvline(xv, color='k', linestyle=':', lw=1.2, alpha=0.9, zorder=11)

    # 
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)
    if secondary_ylim and ax2: ax2.set_ylim(secondary_ylim)
    
    ax.set_xlabel(f"{coord_col} Position", fontsize=12)

    ylabel_dict = {
        'pdf': 'Probability Density',
        'frequency': 'Frequency Density',
        'relative': 'Relative Frequency',
        'count': 'Cell Count',
        'kde': 'Density (KDE)',
        'normalized': 'Normalized Density'
    }
    ax.set_ylabel(ylabel_dict.get(density_method, 'Density'), fontsize=12)
    
    # Y
    if ax2:
        ax2.set_ylabel(f'{ylabel_dict.get(density_method, "Density")} (Secondary)', 
                       fontsize=12)
    
    ax.set_title(f'{coord_col} Distribution of {group_col} Clusters ({density_method.upper()} Method)',
                 fontsize=18, fontweight='bold', pad=20)

    ax.xaxis.set_major_locator(mticker.MultipleLocator(x_tick_interval))
    ax.tick_params(axis='x', rotation=90, labelsize=10)
    
    #  - Y
    if show_legend:
        lines1, labels1 = ax.get_legend_handles_labels()
        if ax2:
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, 
                     bbox_to_anchor=(1.15, 1), loc='upper left')
        else:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    ax.margins(x=0, y=0.02)
    plt.tight_layout()

    # 
    info = {
        "bin_centers": bin_centers,
        "densities": densities,      #  bin 
        "x_dense": x_dense,
        "y_dense": y_dense,          # 
        "intersections": intersections,
        "ax2": ax2,                  # 
        "primary_groups": primary_groups,  # 🆕 
        "secondary_groups": secondary_groups  # 🆕 
    }
    return fig, info

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import matplotlib.ticker as mticker

# ============================================================
# Fully enhanced version: Plot gene expression along AP axis (supports subset filtering and dual Y-axes)
# ============================================================
def plot_genes_along_AP(
    adata,                          # NEW: AnnData object
    gene_list=None,                 # Gene name list (simple mode)
    gene_configs=None,              # NEW: Gene configuration list (advanced mode)
    coord_col='AP',                 # NEW: coordinate column name
    subset_col='sub_leiden',        # NEW: column used for subset filtering
    n_bins=50, 
    show_points=True, 
    smooth=0, 
    figsize=(12, 6), 
    linewidth=2, 
    fill=False,
    fill_alpha=0.3,                 # NEW: fill transparency
    gene_colors=None, 
    gene_linestyles=None,
    vertical_lines=None, 
    vline_color='blue',
    vline_style='--', 
    vline_width=1.5, 
    vline_alpha=0.8,
    show_legend=True,
    xlim=None,                      # NEW: X-axis limits, e.g. (0, 2500)
    ylim=None,                      # NEW: Primary (left) Y-axis limits, e.g. (0,1)
    x_tick_interval=100,            # NEW: X-axis tick interval
    secondary_ylim=None,            # NEW: Secondary (right) Y-axis limits
    normalization='minmax'          # NEW: normalization method: 'minmax', 'zscore', 'none'
):
    

    if gene_configs is None and gene_list is None:
        raise ValueError("Either gene_list or gene_configs must be provided")
    
    if coord_col not in adata.obs.columns:
        raise KeyError(f"Coordinate column not found in obs '{coord_col}'")
    if subset_col not in adata.obs.columns:
        raise KeyError(f"Subset column not found in obs '{subset_col}'")
    
    # 简单模式：Convert gene_list to gene_configs format
    if gene_configs is None:
        gene_configs = [{
            'genes': gene_list,
            'subset_values': None,  # Use all cells
            'use_secondary_y': False,
            'linestyle': None,  # use gene_linestyles
            'color': None       # use gene_colors
        }]
    
    # ========== 2. Create figure and axes ==========
    fig, ax = plt.subplots(figsize=figsize)
    
    # Check whether a secondary Y-axis is needed
    need_secondary_y = any(config.get('use_secondary_y', False) for config in gene_configs)
    ax2 = None
    if need_secondary_y:
        ax2 = ax.twinx()
    
    # ========== 3. Process each configuration group ==========
    all_gene_curves = {}
    
    for config in gene_configs:
        genes = config['genes']
        subset_values = config.get('subset_values', None)
        use_secondary_y = config.get('use_secondary_y', False)
        config_linestyle = config.get('linestyle', '-')
        config_color = config.get('color', None)
        config_n_bins = config.get('n_bins', n_bins)  # Using 配置的n_bins或全局n_bins
        
        # Select current Y-axis
        current_ax = ax2 if (use_secondary_y and ax2 is not None) else ax
        
        # Filter cell subset
        if subset_values is not None:
            # Convert subset_values to string list
            subset_values_str = [str(v) for v in subset_values]
            mask = adata.obs[subset_col].astype(str).isin(subset_values_str)
            adata_subset = adata[mask, :].copy()
            print(f"Using {subset_col}中值为{subset_values} cells, total {mask.sum()} cells, bins={config_n_bins}")
        else:
            adata_subset = adata
            print(f"Use all cells，共{adata_subset.n_obs} cells, bins={config_n_bins}")
        
        # Get and sort coordinate values
        coords = adata_subset.obs[coord_col].values
        order = np.argsort(coords)
        coords_sorted = coords[order]
  
        bins = np.linspace(coords_sorted.min(), coords_sorted.max(), config_n_bins+1)
        bin_indices = np.digitize(coords_sorted, bins) - 1
        bin_indices = np.clip(bin_indices, 0, config_n_bins-1)
        
        # Calculate bin centers
        bin_centers = (bins[:-1] + bins[1:]) / 2
        
        # ========== 4. Calculate expression curve for each gene ==========
        for gene in genes:
            if gene not in adata_subset.var_names:
                print(f"Warning: {gene}  not found, skipped")
                continue
            
            # Get expression
            expr = adata_subset[:, gene].X
            if hasattr(expr, 'toarray'):
                expr = expr.toarray().ravel()
            else:
                expr = expr.ravel()
            expr_sorted = expr[order]
            
            # Calculate mean expression for each bin
            bin_means = np.array([expr_sorted[bin_indices == i].mean() 
                                  if np.any(bin_indices == i) else 0 
                                  for i in range(config_n_bins)])  # Using config_n_bins
            
            # Normalization
            if normalization == 'minmax':
                bin_means_norm = (bin_means - bin_means.min()) / (bin_means.max() - bin_means.min() + 1e-10)
            elif normalization == 'zscore':
                bin_means_norm = (bin_means - bin_means.mean()) / (bin_means.std() + 1e-10)
            elif normalization == 'none':
                bin_means_norm = bin_means
            else:
                raise ValueError(f"未知的Normalization方法: {normalization}")
            
            all_gene_curves[gene] = bin_means_norm
            
            # ========== 5. Plot curves ==========
            # Determine color and linestyle
            if config_color:
                color = config_color
            elif gene_colors and gene in gene_colors:
                color = gene_colors[gene]
            else:
                color = None
            
            if config_linestyle:
                linestyle = config_linestyle
            elif gene_linestyles and gene in gene_linestyles:
                linestyle = gene_linestyles[gene]
            else:
                linestyle = '-'
            
            # Plot curves
            if smooth > 0:
                # Create dense x values for smoothing
                x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), config_n_bins * 10)
                
                try:
                    k = min(int(smooth), len(bin_centers) - 1, 5)
                    spl = make_interp_spline(bin_centers, bin_means_norm, k=k)
                    y_smooth = spl(x_smooth)
                    
         
                    if normalization == 'minmax':
                        y_smooth = np.clip(y_smooth, 0, 1)
                    
                    # Fill area under curve
                    if fill:
                        current_ax.fill_between(x_smooth, y_smooth, alpha=fill_alpha, color=color)
                    
                    # Draw smoothed curve
                    current_ax.plot(x_smooth, y_smooth, linestyle=linestyle, label=gene, 
                                   linewidth=linewidth, color=color, alpha=0.7)
                    
                    # Show original data points
                    if show_points:
                        current_ax.plot(bin_centers, bin_means_norm, 'o', markersize=4, 
                                       alpha=0.5, color=color)
                except Exception as e:
                    print(f"Warning: {gene} Smoothing failed ({str(e)})，Using 原始曲线")
                    marker = 'o-' if show_points else '-'
                    
                    if fill:
                        current_ax.fill_between(bin_centers, bin_means_norm, alpha=fill_alpha, color=color)
                    
                    current_ax.plot(bin_centers, bin_means_norm, marker, label=gene, 
                                   linewidth=linewidth, color=color,
                                   linestyle=linestyle if not show_points else '-',
                                   markersize=4 if show_points else 0, alpha=0.7)
            else:
                # Without smoothing
                marker = 'o-' if show_points else '-'
                
                if fill:
                    current_ax.fill_between(bin_centers, bin_means_norm, alpha=fill_alpha, color=color)
                
                current_ax.plot(bin_centers, bin_means_norm, marker, label=gene, 
                               linewidth=linewidth, color=color,
                               linestyle=linestyle if not show_points else '-',
                               markersize=4 if show_points else 0, alpha=0.7)
    
    # ========== 6. Add vertical reference lines ==========
    if vertical_lines:
        for line_pos in vertical_lines:
            ax.axvline(x=line_pos, color=vline_color, linestyle=vline_style,
                      linewidth=vline_width, alpha=vline_alpha, zorder=10)
    
    # ========== 7. Configure axes ==========

    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)
    if secondary_ylim and ax2:
        ax2.set_ylim(secondary_ylim)
    
    ax.set_xlabel(f'{coord_col} Position', fontsize=12)
    
    ylabel_dict = {
        'minmax': 'Normalized Expression (0-1)',
        'zscore': 'Z-score Normalized Expression',
        'none': 'Expression Level'
    }
    ax.set_ylabel(ylabel_dict.get(normalization, 'Expression'), fontsize=12)
    
    if ax2:
        ax2.set_ylabel(f'{ylabel_dict.get(normalization, "Expression")} (Secondary)', fontsize=12)
    
    ax.set_title(f'Gene Expression along {coord_col} axis', fontsize=14, fontweight='bold')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(x_tick_interval))
    
 
    if show_legend:
        lines1, labels1 = ax.get_legend_handles_labels()
        if ax2:
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, 
                     bbox_to_anchor=(1.15, 1), loc='upper left')
        else:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    ax.margins(x=0, y=0.02)
    plt.tight_layout()
    
    return fig, all_gene_curves

In [ ]:
data= nt[(nt.obs['AP'] >= 480) & (nt.obs['AP'] <= 2100)].copy()

In [ ]:

fig = plot_binned_density(
    data, 
    n_bins=30,
    palette=sub_cluster_palette,
    figsize=(5, 5),
    smooth=2,  
    show_legend=False,
    linewidth=3,
    #vertical_lines=vertical_lines,
    #vline_width=3,
    density_method='count',
    coord_col='DV',
    group_col='leiden_hvg_sub',
    #secondary_y_groups=['e0_3', 'e0_4', 'e0_5'], 
    #dashed_groups=['e0_3', 'e0_4', 'e0_5'],
    #secondary_ylim=(0, 300),
    #x_tick_interval=500,
    fill_alpha=0.5,
    xlim=(0,280),
    #ylim=(0,120),
    primary_y_groups=['e0_0', 'e0_1', 'e0_2', 'e0_3', 'e0_4']
)
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\along_axix\density_plot_nt_leiden_hvg_sub_DV_S4.pdf', dpi=900, bbox_inches='tight')
plt.show()

In [ ]:

gene_colors = {
    "WNT3A":  "#009203",  # 0
    "PAX6":  "#ff002a",  # 1
    "FOXA1":  "#565DFD",  # 2
     #"OLIG2": "#00FFFF",  # 7
    #"CDX2": "#66c102",  # 8
    #"WNT3A": "#8e0119",  # 10
    #"DIO2": "#565DFD",  # 11
    #"KDR": "#ff002a",  # 14
   # "FZD10": "#AB76AE",  # 15
    #"CDH7": "#515151",  # 17
}


subset_values = data.obs['leiden_hvg_sub'].unique().tolist()

fig, curves = plot_genes_along_AP(
    adata=data,
    gene_configs=[
        {
            'genes': list(gene_colors.keys()),
            'subset_values': subset_values,  
            'use_secondary_y': False,
            'linestyle': '-',
            'n_bins': 30
        },
    ],
    coord_col='DV',
    subset_col='leiden_hvg_sub', 
    smooth=3,
    figsize=(5, 5),
    linewidth=3,
    fill=True,
    show_points=False,
    #vertical_lines=vertical_lines,
    vline_width=3.0,
    show_legend=False,
    x_tick_interval=50,
    #secondary_ylim=(0, 1.2),
    gene_colors=gene_colors,
    fill_alpha=0.5,
    xlim=(0,280),
    ylim=(0, 1.1)
)

plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\along_axix\gene_nt_WNT3A_PAX6_FOXA1_DV_S4_S3_RGB.pdf', dpi=900, bbox_inches='tight')
plt.show()

# 4.Extended Data Fig. 4g

In [ ]:
import pandas as pd

df = pd.read_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\cellnest_data_from _alice\15st_2026_02_06_NT_2ST\CellNEST_MYDATA_3D_top20percent.csv")

In [ ]:

m = nt.obs.set_index('cell_id_1')['leiden_hvg_sub']

df['leiden_from'] = df['from_cell'].map(m)


df['leiden_to'] = df['to_cell'].map(m)

global_mean = df.groupby(['ligand', 'receptor'])['attention_score'].transform('mean')
df['specificity_score'] = df['attention_score'] / global_mean

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


real_niche_counts = {
    'e0_0': 4604,
    'e0_1': 2283,
    'e12_0': 2027,
    'e0_2': 1919,
    'e12_1': 1837,
    'e0_3': 1330,
    'e20_0': 1170,
    'e20_1': 1049,
    'e0_4': 765,
    'e12_2': 627,
    'e25_0': 277,  
    'e12_3': 272,  
    'e25_1': 252,  
    'e25_2': 238,  
    'e0_5': 144    
}
counts_series = pd.Series(real_niche_counts)



threshold = 1


high_spec_df = df[df['specificity_score'] > threshold].copy()

spec_count_matrix = high_spec_df.groupby(['leiden_from', 'leiden_to'])['attention_score'].count().unstack(fill_value=0)



spec_density_matrix = spec_count_matrix.div(counts_series, axis=0).fillna(0)

=
plt.figure(figsize=(12, 8))

sns.heatmap(spec_density_matrix, 
            annot=True, 
            fmt='.2f',          
            cmap='Reds',        
            linewidths=.5, 
            linecolor='white',
            cbar_kws={'label': 'Specific Interactions per Sender Cell'}
)

plt.title('Normalized Density of Specific Communication\n(Corrected for Cell Abundance)', fontsize=14)
plt.xlabel('Receiver Niche (To)')
plt.ylabel('Sender Niche (From)')
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\2st_2025_10_07\Figure4\cellnest_LR_PLACENTA_niche_heatmap_all.pdf', dpi=900, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\DEG\cellnest_df.xlsx", index=True)